In [5]:
import sys
import os
import h5py
import numpy as np
import torch

# Add src/ directory to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

In [6]:
from lora import LoRALinear
from processor import LLMTIMEPreprocessor
from data import run_prediction
from qwen import load_qwen

### Untrained Metrics of performance on System IDs

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd

model, tokenizer = load_qwen()

SYSTEM_IDS = [900, 910, 920, 930, 940, 950, 960, 970, 980, 990]
preprocessor = LLMTIMEPreprocessor()
trajectories = h5py.File("../lotka_volterra_data.h5", "r")["trajectories"][:]

# --- Store results
metrics = {
    "System ID": [],
    "MSE": [],
    "MAE": [],
    "R2 Score": []
}

for system_id in SYSTEM_IDS:
    # Load sequence and run forecast
    sequence = trajectories[system_id, :, :]
    input_sequence = sequence[:60]
    true_output_sequence = sequence[60:]

    preprocessed_input = preprocessor.preprocess_sequence(input_sequence)
    tokenized_input = preprocessor.tokenize_sequence(preprocessed_input)
    input_tensor = torch.tensor([tokenized_input]).to(model.device)

    prediction = run_prediction(model, tokenizer, input_tensor, 1)

    # Clip length to match in case decoded output is shorter
    n_pred = min(len(true_output_sequence), len(prediction))
    y_true = true_output_sequence[:n_pred].flatten()
    y_pred = prediction[:n_pred].flatten()

    # Compute metrics
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics["System ID"].append(system_id)
    metrics["MSE"].append(mse)
    metrics["MAE"].append(mae)
    metrics["R2 Score"].append(r2)

# --- Convert to DataFrame and display
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# Optional: Save to CSV
metrics_df.to_csv("../csv/forecast_metrics_untrained.csv", index=False)


   System ID        MSE       MAE   R2 Score
0        900   0.013155  0.091153   0.809718
1        910   0.011695  0.092080   0.573450
2        920   4.179788  1.466199   0.078239
3        930   0.302825  0.424265   0.408528
4        940  17.350063  2.975934 -29.299241
5        950  36.611575  4.860528 -16.351295
6        960   1.639123  0.743050  -0.328788
7        970   0.093856  0.248199  -2.306289
8        980   0.360024  0.458842  -0.773793
9        990   2.905674  1.055886  -6.209285


### Trained (best) Metrics of Performance on System IDs

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd

SYSTEM_IDS = [900, 910, 920, 930, 940, 950, 960, 970, 980, 990]
preprocessor = LLMTIMEPreprocessor()
trajectories = h5py.File("../lotka_volterra_data.h5", "r")["trajectories"][:]

# --- Store results
metrics = {
    "System ID": [],
    "MSE": [],
    "MAE": [],
    "R2 Score": []
}

device = torch.device("cpu")
model, tokenizer = load_qwen()
model.to(device)

# Inject LoRA layers
lora_rank = 4
for layer in model.model.layers:
    layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj, r=lora_rank)
    layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj, r=lora_rank)

# Load trained LoRA weights
model.load_state_dict(torch.load("../models/lora_qwen2.5_final_5000.pth", map_location="cpu"), strict=False)
model.eval()

for system_id in SYSTEM_IDS:
    # Load sequence and run forecast
    sequence = trajectories[system_id, :, :]
    input_sequence = sequence[:60]
    true_output_sequence = sequence[60:]

    preprocessed_input = preprocessor.preprocess_sequence(input_sequence)
    tokenized_input = preprocessor.tokenize_sequence(preprocessed_input)
    input_tensor = torch.tensor([tokenized_input]).to(model.device)

    prediction = run_prediction(model, tokenizer, input_tensor, 1)

    # Clip length to match in case decoded output is shorter
    n_pred = min(len(true_output_sequence), len(prediction))
    y_true = true_output_sequence[:n_pred].flatten()
    y_pred = prediction[:n_pred].flatten()

    # Compute metrics
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics["System ID"].append(system_id)
    metrics["MSE"].append(mse)
    metrics["MAE"].append(mae)
    metrics["R2 Score"].append(r2)

# --- Convert to DataFrame and display
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# Optional: Save to CSV
metrics_df.to_csv("../csv/forecast_metrics_trained_5000.csv", index=False)


   System ID       MSE       MAE  R2 Score
0        900  0.413269  0.459270 -4.977783
1        910  0.072717  0.215638 -1.652201
2        920  2.748286  1.438000  0.393926
3        930  0.896017  0.801765 -0.750083
4        940  0.945101  0.610309 -0.650475
5        950  1.942634  0.991399  0.079329
6        960  3.535460  1.231617 -1.866090
7        970  0.088686  0.240847 -2.124143
8        980  0.496614  0.534075 -1.446752
9        990  1.219205  0.721532 -2.024977


### CTX Metrics of Performance on System IDs

In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import torch
import numpy as np

# --- Store results
metrics = {
    "System ID": [],
    "Context Length": [],
    "MSE": [],
    "MAE": [],
    "R2 Score": []
}

device = torch.device("cpu")
SYSTEM_IDS = [900, 920, 940, 950, 970]
CTX_MODELS = {
    128: "../models/lora_qwen2.5_ctx128_testing.pth",
    512: "../models/lora_qwen2.5_ctx512_testing.pth",
    768: "../models/lora_qwen2.5_ctx768_testing.pth"
}

# Load shared components
model_base, tokenizer = load_qwen()
model_base.to(device)

preprocessor = LLMTIMEPreprocessor()
trajectories = h5py.File("../lotka_volterra_data.h5", "r")["trajectories"][:]
SPLIT_POINT = 60

def run_prediction(model, tokenizer, input_tensor):
    attention_mask = (input_tensor != tokenizer.pad_token_id).long()
    with torch.no_grad():
        output_tokens = model.generate(
            input_tensor, 
            attention_mask=attention_mask,
            max_new_tokens=1000,
            min_length=1000
        )
    generated_tokens = output_tokens[0].tolist()[len(input_tensor[0]):]
    decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    try:
        parsed = np.array([
            list(map(float, pair.split(",")))
            for pair in decoded_output.split(";") if "," in pair
        ])
        return parsed[:40] * 10 if parsed.shape[0] >= 40 else np.zeros((40, 2))
    except ValueError:
        return np.zeros((40, 2))

# Evaluate each model
for ctx_length, model_path in CTX_MODELS.items():
    # Reload base model each time for clean injection
    model, _ = load_qwen()
    model.to(device)
    model.eval()

    # Inject LoRA
    for layer in model.model.layers:
        layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj, r=8)
        layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj, r=8)

    model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=False)
    model.eval()

    for system_id in SYSTEM_IDS:
        sequence = trajectories[system_id, :, :]
        input_sequence = sequence[:SPLIT_POINT]
        true_output_sequence = sequence[SPLIT_POINT:]

        preprocessed_input = preprocessor.preprocess_sequence(input_sequence)
        tokenized_input = preprocessor.tokenize_sequence(preprocessed_input)
        input_tensor = torch.tensor([tokenized_input]).to(model.device)

        prediction = run_prediction(model, tokenizer, input_tensor)

        n_pred = min(len(true_output_sequence), len(prediction))
        y_true = true_output_sequence[:n_pred].flatten()
        y_pred = prediction[:n_pred].flatten()

        # Metrics
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        metrics["System ID"].append(system_id)
        metrics["Context Length"].append(ctx_length)
        metrics["MSE"].append(mse)
        metrics["MAE"].append(mae)
        metrics["R2 Score"].append(r2)

# --- Save Results
metrics_df = pd.DataFrame(metrics)
print(metrics_df)
metrics_df.to_csv("../csv/forecast_metrics_by_context.csv", index=False)


    System ID  Context Length         MSE        MAE   R2 Score
0         900             128    0.326417   0.507231  -3.721501
1         920             128   14.279890   3.121750  -2.149117
2         940             128    5.138399   1.789814  -7.973430
3         950             128    0.453862   0.443624   0.784901
4         970             128    0.056366   0.180561  -0.985601
5         900             512    0.038370   0.149452   0.445000
6         920             512    0.530201   0.658000   0.883076
7         940             512    6.691081   1.994684 -10.684953
8         950             512  158.452394  10.700654 -74.095219
9         970             512    0.056366   0.180561  -0.985601
10        900             768    0.122484   0.287165  -0.771690
11        920             768   14.279890   3.121750  -2.149117
12        940             768   15.821130   2.833434 -26.629192
13        950             768    4.098806   1.410243  -0.942544
14        970             768    0.05636